# 急诊双结局课程实验

本笔记本按课程精简方案执行五种算法、统一分层五折、双结局AUC与耗时比较。依据五折平均Macro AUC选择算法后，使用其OOF预测确定两个Youden阈值，再在全部70万条训练记录上重拟合，生成同源留出及Kaggle预测。数据为公开合成表格，统计单位为就诊记录。

## 环境、数据版本与检查点

模型使用固定参数，CPU线程上限4。训练时间覆盖训练预处理与两个分类器拟合，预测时间覆盖验证转换与两个结局的概率预测。各阶段保留签名、模型、原始时间和预测检查点。

In [ ]:
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import fcntl
import gc
import hashlib
import importlib.metadata
import json
import os
import threading
import time
import warnings

import joblib
import nbformat
import numpy as np
import pandas as pd
import psutil
from IPython.display import HTML, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

PROJECT_ROOT=Path(os.environ.get("ML_PROJECT_ROOT",".")).resolve()
config=json.loads((PROJECT_ROOT/"configs/course_study_run.json").read_text())
contract=json.loads((PROJECT_ROOT/"configs/baseline_features.json").read_text())
parameters=json.loads((PROJECT_ROOT/"configs/baseline_model_parameters.json").read_text())
pointer=json.loads((PROJECT_ROOT/"configs/active_data_release.json").read_text())
release=json.loads(Path(pointer["release_json"]).read_text())
RUN=PROJECT_ROOT/"outputs/course_study"/config["run_id"]
for name in ["checkpoints","models","predictions","figures"]:
    (RUN/name).mkdir(parents=True,exist_ok=True)
lock_stream=(RUN/"execution.lock").open("a+")
fcntl.flock(lock_stream.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
BASE=contract["allowed_columns"]
CATEGORICAL=contract["categorical_columns"]
targets=config["targets"]
identifier="encounter_id"
methods=config["models"]
seed=config["cv"]["seed"]
threads=config["time_metrics"]["threads"]
threadpool_limits(limits=threads)
MODEL_CLASSES={
    "LogisticRegression":LogisticRegression,"RandomForest":RandomForestClassifier,
    "XGBoost":XGBClassifier,"LightGBM":LGBMClassifier,"CatBoost":CatBoostClassifier,
}
versions={name:importlib.metadata.version(name) for name in [
    "numpy","pandas","scipy","scikit-learn","lightgbm","xgboost","catboost","pyarrow",
]}


def atomic_json(path,value):
    path=Path(path)
    temporary=path.with_name("."+path.name+".tmp")
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""):
            digest.update(block)
    return digest.hexdigest()


def hash_value(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str).encode()).hexdigest()


def atomic_model(path,value):
    temporary=path.with_name("."+path.name+".tmp")
    joblib.dump(value,temporary,compress=3)
    temporary.replace(path)


fit_records=json.loads((RUN/"fit_records.json").read_text()) if (RUN/"fit_records.json").exists() else {}
attempts=json.loads((RUN/"fit_attempts.json").read_text()) if (RUN/"fit_attempts.json").exists() else []
for attempt in attempts:
    if attempt["status"]=="running":
        attempt["status"]="interrupted_before_restart"
print("课程实验",config["run_id"],flush=True)
print("线程上限",threads)


## 70万条训练记录与共同五折

合并既有审计版本中的全部train记录，并按原CSV标识顺序恢复。只载入训练资料的标签。保留93个允许字段，按两个标签的四种组合分层。

In [ ]:
def prepare_base_features(raw):
    values={}
    for column in BASE:
        if column in CATEGORICAL:
            values[column]=raw[column].astype("string").fillna("__MISSING__").astype(object)
        else:
            values[column]=pd.to_numeric(raw[column],errors="raise").to_numpy(dtype=np.float32,na_value=np.nan)
    return pd.DataFrame(values,index=raw.index).reset_index(drop=True)


processed=Path(release["processed_directory"])
for filename in ["development.parquet","temporal_test_locked.parquet"]:
    if sha256_file(processed/filename)!=release["artifact_sha256"][filename]:
        raise ValueError("训练资料版本校验失败")
raw_train=PROJECT_ROOT/"data/raw/train.csv"
expected_source=next(value for value in release["source_files"] if value["file"]=="train.csv")
if sha256_file(raw_train)!=expected_source["sha256"]:
    raise ValueError("原始训练文件校验失败")
raw_ids=pd.read_csv(raw_train,usecols=[identifier],dtype={identifier:"string"})[identifier].astype(str)
raw=pd.concat([
    pd.read_parquet(processed/filename,columns=[identifier]+BASE+targets)
    for filename in ["development.parquet","temporal_test_locked.parquet"]
],ignore_index=True)
raw[identifier]=raw[identifier].astype(str)
if raw[identifier].duplicated().any() or len(raw)!=700000 or set(raw[identifier])!=set(raw_ids):
    raise ValueError("训练记录标识或数量不一致")
raw=raw.set_index(identifier).loc[raw_ids].reset_index()
IDS=raw[identifier].to_numpy()
Y=raw[targets].to_numpy(dtype=np.int8)
if not np.isin(Y,[0,1]).all():
    raise ValueError("目标标签存在非法值")
X=prepare_base_features(raw)
del raw,raw_ids
gc.collect()
folds=np.full(len(X),-1,dtype=np.int8)
strata=Y[:,0]*2+Y[:,1]
for fold,(_,validation) in enumerate(StratifiedKFold(n_splits=5,shuffle=True,random_state=seed).split(X,strata)):
    folds[validation]=fold
if (folds<0).any() or len(BASE)!=93 or len(CATEGORICAL)!=14:
    raise AssertionError("共同划分或特征定义不完整")
fold_frame=pd.DataFrame({identifier:IDS,"fold":folds,targets[0]:Y[:,0],targets[1]:Y[:,1]})
fold_path=RUN/"development_folds.parquet"
if fold_path.exists():
    pd.testing.assert_frame_equal(pd.read_parquet(fold_path),fold_frame)
else:
    fold_frame.to_parquet(fold_path,index=False)
model_signature=hash_value({
    "config":config,"features":contract,"parameters":parameters,"versions":versions,
    "source_bundle":release["source_bundle_sha256"],"fold_sha256":sha256_file(fold_path),
})
configuration={"signature":model_signature,"config":config,"features":contract,"parameters":parameters,"versions":versions,"release":release}
if (RUN/"configuration.json").exists():
    if json.loads((RUN/"configuration.json").read_text())["signature"]!=model_signature:
        raise ValueError("已有实验检查点与当前配置不一致")
else:
    atomic_json(RUN/"configuration.json",configuration)
atomic_json(RUN/"data_summary.json",{
    "rows":len(X),"features":len(BASE),"categorical_features":len(CATEGORICAL),
    "events":{name:int(Y[:,index].sum()) for index,name in enumerate(targets)},
    "fold_rows":pd.Series(folds).value_counts().sort_index().to_dict(),
})
print("共同五折",pd.Series(folds).value_counts().sort_index().to_dict(),flush=True)


## 预处理、模型与计时

复用已经验证的算法适配预处理与模型类别。每个训练分区先拟合一次预处理，再分别拟合两个目标。检查点缓存保存初次实测的时间。

In [ ]:
def encode_inputs(fitting, prediction_frames, columns, algorithm):
    categorical = [column for column in CATEGORICAL if column in columns]
    numeric = [column for column in columns if column not in categorical]
    if algorithm in {"LightGBM", "CatBoost"}:
        frequent = {}
        levels = {}
        for column in categorical:
            counts = fitting[column].value_counts()
            frequent[column] = set(counts[counts >= 50].index)
            levels[column] = sorted(frequent[column] | {"__MISSING__", "__OTHER__"})
        state = {
            "kind":"native", "algorithm":algorithm, "columns":columns,
            "categorical":categorical, "frequent":frequent, "levels":levels,
        }
        def transform(frame):
            result = frame[columns].copy()
            for column in categorical:
                result[column] = result[column].where(
                    result[column].isin(frequent[column]), "__OTHER__"
                )
                if algorithm == "LightGBM":
                    result[column] = pd.Categorical(
                        result[column], categories=levels[column]
                    )
            return result
        fitting_matrix = transform(fitting)
        prediction_matrices = {
            key: transform(frame) for key, frame in prediction_frames.items()
        }
    else:
        steps = [SimpleImputer(strategy="median", keep_empty_features=True)]
        if algorithm == "LogisticRegression":
            steps.append(StandardScaler())
        encoder = ColumnTransformer([
            ("numeric", make_pipeline(*steps), numeric),
            ("categorical", OneHotEncoder(
                min_frequency=50, handle_unknown="infrequent_if_exist",
                sparse_output=True, dtype=np.float32,
            ), categorical),
        ], sparse_threshold=1.0)
        fitting_matrix = encoder.fit_transform(fitting[columns]).astype(np.float32)
        prediction_matrices = {
            key: encoder.transform(frame[columns]).astype(np.float32)
            for key, frame in prediction_frames.items()
        }
        state = {
            "kind":"one_hot", "algorithm":algorithm,
            "columns":columns, "categorical":categorical, "encoder":encoder,
        }
    return fitting_matrix, prediction_matrices, state

def transform_saved_preprocessing(frame,state):
    columns = state["columns"]
    if state["kind"] == "one_hot":
        return state["encoder"].transform(frame[columns]).astype(np.float32)
    result = frame[columns].copy()
    for column in state["categorical"]:
        result[column] = result[column].where(
            result[column].isin(state["frequent"][column]),"__OTHER__"
        )
        if state["algorithm"] == "LightGBM":
            result[column] = pd.Categorical(
                result[column],categories=state["levels"][column]
            )
    return result

class PeakMemory:
    def __init__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self.finished = threading.Event()
        self.worker = threading.Thread(target=self.sample, daemon=True)

    def sample(self):
        while not self.finished.wait(0.05):
            self.peak = max(self.peak, self.process.memory_info().rss)

    def __enter__(self):
        self.worker.start()
        return self

    def __exit__(self, exception_type, exception_value, traceback):
        self.peak = max(self.peak, self.process.memory_info().rss)
        self.finished.set()
        self.worker.join()


def fit_pair(context,method,positions):
    directory=RUN/"checkpoints"/context
    directory.mkdir(parents=True,exist_ok=True)
    pair_signature=hash_value({"signature":model_signature,"context":context,"method":method,"positions":hashlib.sha256(np.asarray(positions,dtype=np.int64).tobytes()).hexdigest()})
    state_path=directory/"preprocessing.joblib"
    state_meta_path=directory/"preprocessing.json"
    matrix=None
    fitting=None
    if state_meta_path.exists() and state_path.exists():
        state_meta=json.loads(state_meta_path.read_text())
        if state_meta["signature"]!=pair_signature or sha256_file(state_path)!=state_meta["sha256"]:
            raise ValueError("预处理检查点不一致")
        state=joblib.load(state_path)
    else:
        fitting=X.iloc[positions]
        started=time.perf_counter()
        matrix,_,state=encode_inputs(fitting,{},BASE,method)
        preprocessing_seconds=time.perf_counter()-started
        atomic_model(state_path,state)
        state_meta={"signature":pair_signature,"preprocessing_seconds":preprocessing_seconds,"sha256":sha256_file(state_path)}
        atomic_json(state_meta_path,state_meta)
    statistics=[]
    for target_index,target in enumerate(targets):
        metadata_path=directory/f"target_{target_index}.json"
        model_path=directory/f"target_{target_index}.joblib"
        if metadata_path.exists() and model_path.exists():
            metadata=json.loads(metadata_path.read_text())
            if metadata["signature"]!=pair_signature or sha256_file(model_path)!=metadata["model_sha256"]:
                raise ValueError("分类器检查点不一致")
        else:
            if matrix is None:
                if fitting is None:
                    fitting=X.iloc[positions]
                matrix=transform_saved_preprocessing(fitting,state)
            atomic_json(RUN/"progress.json",{
                "status":"running","phase":"fitting","context":context,"method":method,
                "target":target,"completed_fits":len(fit_records),"expected_fits":52,
            })
            model=MODEL_CLASSES[method](**deepcopy(parameters[method+"__"+target]))
            attempt={"context":context,"method":method,"target":target,"status":"running","started_at":datetime.now(ZoneInfo("Asia/Shanghai")).isoformat()}
            attempts.append(attempt)
            atomic_json(RUN/"fit_attempts.json",attempts)
            started=time.perf_counter()
            try:
                with PeakMemory() as memory,warnings.catch_warnings(record=True) as captured:
                    warnings.simplefilter("always")
                    if method=="CatBoost":
                        model.fit(matrix,Y[positions,target_index],cat_features=state["categorical"])
                    else:
                        model.fit(matrix,Y[positions,target_index])
                fitting_seconds=time.perf_counter()-started
                if list(model.classes_)!=[0,1]:
                    raise AssertionError("分类器类别顺序不一致")
                attempt.update(status="completed",fitting_seconds=fitting_seconds)
                atomic_json(RUN/"fit_attempts.json",attempts)
                atomic_model(model_path,model)
                metadata={
                    "signature":pair_signature,"context":context,"method":method,"target":target,
                    "fitting_rows":int(len(positions)),"fitting_seconds":fitting_seconds,
                    "peak_fitting_process_rss_mib":memory.peak/1024**2,
                    "warnings":[str(value.message) for value in captured],
                    "model_path":str(model_path),"model_sha256":sha256_file(model_path),
                }
                atomic_json(metadata_path,metadata)
            except Exception as error:
                attempt.update(status="failed",error_type=type(error).__name__)
                atomic_json(RUN/"fit_attempts.json",attempts)
                raise
            del model
            gc.collect()
        fit_records[context+"__"+str(target_index)]=metadata
        atomic_json(RUN/"fit_records.json",fit_records)
        statistics.append(metadata)
    del matrix,fitting
    gc.collect()
    return {"context":context,"method":method,"directory":directory,"state":state,"preprocessing":state_meta,"statistics":statistics,"training_seconds":state_meta["preprocessing_seconds"]+sum(value["fitting_seconds"] for value in statistics)}


def predict_pair(pair,frame,name,ids):
    directory=pair["directory"]
    prediction_path=directory/f"{name}_probabilities.npz"
    metadata_path=directory/f"{name}_prediction.json"
    prediction_signature=hash_value({
        "signature":model_signature,"name":name,
        "models":[value["model_sha256"] for value in pair["statistics"]],
        "preprocessing":pair["preprocessing"]["sha256"],
        "ids":hashlib.sha256(np.asarray(ids,dtype=str).tobytes()).hexdigest(),
    })
    if prediction_path.exists() and metadata_path.exists():
        metadata=json.loads(metadata_path.read_text())
        if metadata["signature"]!=prediction_signature or sha256_file(prediction_path)!=metadata["sha256"]:
            raise ValueError("概率检查点不一致")
        with np.load(prediction_path,allow_pickle=False) as saved:
            probability=saved["probability"]
    else:
        models=[joblib.load(Path(value["model_path"])) for value in pair["statistics"]]
        started=time.perf_counter()
        matrix=transform_saved_preprocessing(frame,pair["state"])
        probability=np.empty((len(frame),2),dtype=np.float64)
        for target_index,model in enumerate(models):
            if pair["method"]=="CatBoost":
                probability[:,target_index]=model.predict_proba(matrix,thread_count=threads)[:,1]
            else:
                probability[:,target_index]=model.predict_proba(matrix)[:,1]
        seconds=time.perf_counter()-started
        if not np.isfinite(probability).all() or (probability<0).any() or (probability>1).any():
            raise ValueError("预测概率存在非法值")
        temporary=prediction_path.with_name("."+prediction_path.stem+".tmp.npz")
        np.savez_compressed(temporary,probability=probability)
        temporary.replace(prediction_path)
        metadata={"signature":prediction_signature,"rows":len(frame),"prediction_seconds":seconds,"seconds_per_10000":seconds*10000/len(frame),"sha256":sha256_file(prediction_path)}
        atomic_json(metadata_path,metadata)
        del models,matrix
        gc.collect()
    if probability.shape!=(len(frame),2):
        raise AssertionError("预测记录数不一致")
    return probability,metadata


## 五种算法的统一五折比较

每个留出折只报告两个目标AUC、宏平均AUC和时间。OOF概率在算法选择后用于阈值阶段。

In [ ]:
oof={method:np.full((len(X),2),np.nan,dtype=np.float64) for method in methods}
cv_rows=[]
for fold in range(5):
    fitting_positions=np.flatnonzero(folds!=fold)
    validation_positions=np.flatnonzero(folds==fold)
    for method in methods:
        context=f"fold{fold}_{method}"
        pair=fit_pair(context,method,fitting_positions)
        probability,prediction_stats=predict_pair(pair,X.iloc[validation_positions],"validation",IDS[validation_positions])
        oof[method][validation_positions]=probability
        auc=[roc_auc_score(Y[validation_positions,target],probability[:,target]) for target in range(2)]
        cv_rows.append({
            "fold":fold,"method":method,"training_rows":len(fitting_positions),"validation_rows":len(validation_positions),
            "auc_A":auc[0],"auc_R":auc[1],"macro_auc":float(np.mean(auc)),
            "training_seconds":pair["training_seconds"],
            "preprocessing_seconds":pair["preprocessing"]["preprocessing_seconds"],
            "classifier_fitting_seconds":sum(value["fitting_seconds"] for value in pair["statistics"]),
            "prediction_seconds":prediction_stats["prediction_seconds"],
            "prediction_seconds_per_10000":prediction_stats["seconds_per_10000"],
        })
        pd.DataFrame(cv_rows).to_csv(RUN/"cv_fold_metrics.csv",index=False)
        print("完成",context,"Macro AUC",round(float(np.mean(auc)),6),flush=True)
        del pair,probability
        gc.collect()
for method,value in oof.items():
    if not np.isfinite(value).all():
        raise AssertionError("折外预测不完整")
cv_frame=pd.DataFrame(cv_rows)
cv_summary=cv_frame.groupby("method",as_index=False).agg(
    auc_A_mean=("auc_A","mean"),auc_A_sd=("auc_A","std"),
    auc_R_mean=("auc_R","mean"),auc_R_sd=("auc_R","std"),
    macro_auc_mean=("macro_auc","mean"),macro_auc_sd=("macro_auc","std"),
    training_seconds_mean=("training_seconds","mean"),training_seconds_sd=("training_seconds","std"),
    prediction_seconds_per_10000_mean=("prediction_seconds_per_10000","mean"),
    prediction_seconds_per_10000_sd=("prediction_seconds_per_10000","std"),
).sort_values(["macro_auc_mean","training_seconds_mean"],ascending=[False,True])
cv_summary.to_csv(RUN/"cv_summary.csv",index=False)
selected=cv_summary.iloc[0]["method"]
oof_table=fold_frame.copy()
for method,values in oof.items():
    for target_index,target in enumerate(targets):
        oof_table[method+"__"+target]=values[:,target_index]
oof_table.to_parquet(RUN/"predictions/development_oof.parquet",index=False)
atomic_json(RUN/"selected_algorithm.json",{
    "method":selected,"selection":"five-fold mean macro ROC-AUC, exact tie then lower mean training seconds",
    "macro_auc_mean":float(cv_summary.iloc[0]["macro_auc_mean"]),
    "selected_before_holdout_evaluation":True,
})
print("五折选定算法",selected,flush=True)


## 选定算法的两个Youden阈值

在全体训练记录的选定算法OOF概率上分别最大化J。使用整数计数比较相同J值，候选包含0、1及实际概率值，并列时取较低有限阈值。

In [ ]:
def youden_threshold(labels,probability):
    labels=np.asarray(labels,dtype=np.int64)
    candidates=np.unique(np.r_[0.0,probability,1.0])
    order=np.argsort(probability,kind="stable")
    sorted_scores=probability[order]
    cumulative=np.r_[0,np.cumsum(labels[order],dtype=np.int64)]
    left=np.searchsorted(sorted_scores,candidates,side="left")
    positive=int(labels.sum())
    negative=len(labels)-positive
    if positive==0 or negative==0:
        raise ValueError("Youden阈值需要两个类别")
    tp=positive-cumulative[left]
    fp=len(labels)-left-tp
    numerators=tp*negative-fp*positive
    winner=np.flatnonzero(numerators==numerators.max())[0]
    return {"threshold":float(candidates[winner]),"youden":float(numerators[winner]/(positive*negative)),"positive_records":positive,"negative_records":negative,"selection":"pooled selected-algorithm OOF; lowest threshold on exact J ties"}


check_y=np.array([0,1,0,1,1,0,0,1])
check_p=np.array([0.1,0.15,0.15,0.3,0.4,0.5,0.7,0.9])
check=youden_threshold(check_y,check_p)
check_grid=np.unique(np.r_[0.0,check_p,1.0])
check_j=np.array([np.mean(check_p[check_y==1]>=t)-np.mean(check_p[check_y==0]>=t) for t in check_grid])
assert check["threshold"]==float(check_grid[np.flatnonzero(np.isclose(check_j,check_j.max(),rtol=0,atol=1e-14))[0]])
threshold_records={target:youden_threshold(Y[:,index],oof[selected][:,index]) for index,target in enumerate(targets)}
atomic_json(RUN/"youden_thresholds.json",threshold_records)
print("选定算法阈值",{name:value["threshold"] for name,value in threshold_records.items()},flush=True)
del oof,oof_table
gc.collect()


## 全量重拟合与锁定

使用全部70万条训练记录重新拟合选定算法的两个分类器，再保存模型、预处理和阈值的完整锁定记录。

In [ ]:
final_pair=fit_pair("final_"+selected,selected,np.arange(len(X)))
locked={
    "signature":model_signature,"selected_algorithm":selected,
    "locked_at":datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
    "training_records":len(X),"thresholds":threshold_records,
    "preprocessing_path":str(final_pair["directory"]/"preprocessing.joblib"),
    "preprocessing_sha256":final_pair["preprocessing"]["sha256"],
    "classifiers":final_pair["statistics"],
    "cv_summary_sha256":sha256_file(RUN/"cv_summary.csv"),
    "oof_sha256":sha256_file(RUN/"predictions/development_oof.parquet"),
    "prior_supplemental_exploratory_evaluation":True,
    "config":config,
}
freeze_path=RUN/"frozen_protocol.json"
if freeze_path.exists():
    previous=json.loads(freeze_path.read_text())
    if previous["signature"]!=model_signature or previous["classifiers"]!=locked["classifiers"] or previous["thresholds"]!=threshold_records:
        raise ValueError("已有锁定记录与当前最终模型不一致")
    locked=previous
else:
    atomic_json(freeze_path,locked)
frozen_digest=sha256_file(freeze_path)
atomic_json(RUN/"frozen_protocol_receipt.json",{"path":str(freeze_path),"sha256":frozen_digest})
print("模型已锁定",frozen_digest,flush=True)
del X
gc.collect()


## 同源留出与Kaggle概率

只读取相应输入字段，生成两个目标的概率。Kaggle CSV按sample_submission中的标识顺序输出。

In [ ]:
prediction_receipts={}
for partition,filename,expected_rows in [
    ("S","supplemental_locked.parquet",100000),
    ("K","kaggle_test.parquet",200000),
]:
    source_path=processed/filename
    if sha256_file(source_path)!=release["artifact_sha256"][filename]:
        raise ValueError("评价输入文件版本不一致")
    raw=pd.read_parquet(source_path,columns=[identifier]+BASE)
    ids=raw[identifier].astype(str).to_numpy()
    if len(ids)!=expected_rows or len(set(ids))!=len(ids) or set(ids)&set(IDS):
        raise ValueError("评价标识或数量不一致")
    features=prepare_base_features(raw)
    probability,prediction_stats=predict_pair(final_pair,features,partition,ids)
    frame=pd.DataFrame({identifier:ids})
    for target_index,target in enumerate(targets):frame[target]=probability[:,target_index]
    path=RUN/"predictions"/("holdout_probabilities.parquet" if partition=="S" else "kaggle_probabilities.parquet")
    frame.to_parquet(path,index=False)
    prediction_receipts[partition]={
        "path":str(path),"rows":len(frame),"sha256":sha256_file(path),
        "frozen_protocol_sha256":frozen_digest,"prediction_statistics":prediction_stats,
    }
    if partition=="K":
        sample_path=PROJECT_ROOT/"data/raw/sample_submission.csv"
        sample=pd.read_csv(sample_path,dtype={identifier:str})
        if sample.columns.tolist()!=config["submission_columns"]:
            raise ValueError("提交列与官方样例不一致")
        submission=sample[[identifier]].merge(frame,on=identifier,how="left",validate="one_to_one")
        if len(submission)!=200000 or submission[targets].isna().any().any():
            raise AssertionError("提交文件覆盖不完整")
        submission_path=RUN/"submission.csv"
        submission.to_csv(submission_path,index=False,float_format="%.12g")
        atomic_json(RUN/"submission_receipt.json",{
            "path":str(submission_path),"rows":len(submission),"columns":submission.columns.tolist(),
            "sha256":sha256_file(submission_path),"selected_algorithm":selected,
            "frozen_protocol_sha256":frozen_digest,
        })
    atomic_json(RUN/"prediction_receipts.json",prediction_receipts)
    print("已生成概率",partition,len(frame),flush=True)
    del raw,features,probability,frame
    gc.collect()
if len(fit_records)!=52:
    raise AssertionError("完成的分类器拟合数量与方案不一致")
if sha256_file(freeze_path)!=frozen_digest:
    raise ValueError("预测期间锁定方案发生变化")
summary={
    "status":"completed","run_id":config["run_id"],"training_records":len(IDS),
    "selected_algorithm":selected,"classifier_fits":len(fit_records),
    "frozen_protocol_sha256":frozen_digest,"thresholds":threshold_records,
    "prediction_receipts":prediction_receipts,"run_directory":str(RUN),
    "fit_warnings":sum(len(value["warnings"]) for value in fit_records.values()),
    "fit_attempts":len(attempts),"final_training_seconds":final_pair["training_seconds"],
}
atomic_json(RUN/"summary.json",summary)
atomic_json(RUN/"progress.json",{"status":"completed",**summary})
print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)


## 开发比较结果表

下表保留全部五种算法的五折结果，表题位于下方。正式留出评价由Notebook 07读取冻结预测完成。

In [ ]:
def scientific_table(frame, caption, precision=4):
    styled = (
        frame.style
        .hide(axis="index")
        .format(precision=precision, na_rep="—")
        .set_caption(caption)
        .set_table_styles([
            {"selector": "", "props": [
                ("border-collapse", "collapse"),
                ("border-top", "1.5pt solid black"),
                ("border-bottom", "1.5pt solid black"),
                ("font-family", "'Times New Roman', SimSun, serif"),
            ]},
            {"selector": "thead th", "props": [
                ("border-bottom", "0.75pt solid black"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("background-color", "white"),
                ("color", "black"),
                ("padding", "6px 10px"),
            ]},
            {"selector": "tbody td", "props": [
                ("border", "none"),
                ("padding", "5px 10px"),
                ("background-color", "white"),
                ("color", "black"),
            ]},
            {"selector": "caption", "props": [
                ("caption-side", "bottom"),
                ("text-align", "center"),
                ("color", "black"),
                ("padding-top", "7px"),
            ]},
        ])
    )
    display(HTML(styled.to_html()))
scientific_table(cv_summary,"表1 课程实验五种算法的五折AUC与双结局训练及预测耗时",precision=5)
